In [1]:
# %% 1. Imports
import sys
from pathlib import Path

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

import dataclasses

import pandas as pd

from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    ArrivalsPhase,
    DeparturePhysicsPhase,
    Environment,
    EnvironmentConfig,
    HistoricalLatentDemandPhase,
    HistoricalODStructurePhase,
    HistoricalTripSamplingPhase,
)
from gbp.consumers.simulator.built_in_phases import LatentDemandInflatorPhase
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.phases import Schedule
from gbp.consumers.simulator.tasks.rebalancer import RebalancerTask
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

imports_loaded = True

In [2]:
# %% 2. Build resolved mock model with trucks
mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved_full = build_model(raw)

# Drop supply: observed_flow already encodes trip events on its target side,
# pre-populating in_transit from supply would double-count arrivals.  Same
# rationale as notebook 09 cell 3.
resolved = dataclasses.replace(resolved_full, supply=None)
demand_multiplier = 2.0
rebalance_every_n_periods = 6
rebalancer_commodity = "electric_bike"

2026-05-03 22:34:23 [info     ] load_start                     loader=graph_core
2026-05-03 22:34:24 [debug    ] source_validated               loader=graph_core
2026-05-03 22:34:24 [debug    ] observed_flow_built            loader=graph_core rows=305
2026-05-03 22:34:24 [debug    ] observed_inventory_built       loader=graph_core rows=60
2026-05-03 22:34:24 [info     ] load_done                      facilities=12 loader=graph_core


In [3]:
# %% 4. Treatment run: same pipeline + RebalancerTask
rebalancer_task = RebalancerTask(
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
    commodity_category=rebalancer_commodity,
)
treatment_phases = [
    # Calculate latent_departures and latent_arrivals as intermidiate part of state saved as latent_demand
    HistoricalLatentDemandPhase(),
    # multiplying latent_departures and latent_arrivals on demand_multiplier
    LatentDemandInflatorPhase(multiplier=demand_multiplier),
    # Calculating OD - probability of target distribution of trips for each source. Writing it into intermidiates state as od_probabilities
    HistoricalODStructurePhase(),
    # Aplying constraints for inventory. Demand could be > then inventory. We need to resize. But in this case we will lose something. We also need to log it.
    DeparturePhysicsPhase(mode="strict"),
    # Just rewriting historical flows to current "simulated" flows
    HistoricalTripSamplingPhase(),
    # Updating inventory based on completed trips. Keepind remaining trips in intransit. Wirting arrived flows to events. Updating resources - this is not clear since during the simulation we are not using resources.
    ArrivalsPhase(),

    DispatchPhase(
        rebalancer_task,
        schedule=Schedule.every_n(rebalance_every_n_periods),
    ),
]

env = Environment(
    resolved,
    EnvironmentConfig(phases=treatment_phases, seed=42, scenario_id="treatment"),
)

while not env.is_done:
    if env.is_done:
        raise StopIteration("All periods have been processed.")

    period = env._periods[env._period_cursor]

    for phase in env._config.phases:
        if phase.should_run(period):
            result = phase.execute(env._state, env._resolved, period)
            env._state = result.state
            env._log.record_events(result, phase.name, period)

    env._log.record_period(env._state, period)

    # Advance to next period
    env._period_cursor += 1
    if not env.is_done:
        next_period = env._periods[env._period_cursor]
        env._state = env._state.advance_period(
            next_period_index=next_period.period_index,
            next_period_id=next_period.period_id,
        )
treatment_log = env._log
treatment_tables = treatment_log.to_dataframes()

In [4]:
treatment_tables['simulation_resource_log']

,period_index,period_id,resource_id,resource_category,current_facility_id,status,available_at_period,home_facility_id
0,0,p0,truck_1,rebalancing_truck,station_3,in_transit,1,depot_1
1,0,p0,truck_2,rebalancing_truck,depot_1,available,None,depot_1
2,0,p0,truck_3,rebalancing_truck,depot_1,available,None,depot_1
3,1,p1,truck_1,rebalancing_truck,station_3,available,None,depot_1
4,1,p1,truck_2,rebalancing_truck,depot_1,available,None,depot_1
5,1,p1,truck_3,rebalancing_truck,depot_1,available,None,depot_1
6,2,p2,truck_1,rebalancing_truck,station_3,available,None,depot_1
7,2,p2,truck_2,rebalancing_truck,depot_1,available,None,depot_1
8,2,p2,truck_3,rebalancing_truck,depot_1,available,None,depot_1


In [20]:
env._state.resources

,resource_id,resource_category,home_facility_id,current_facility_id,status,available_at_period
0,truck_1,rebalancing_truck,depot_1,station_3,available,None
1,truck_2,rebalancing_truck,depot_1,depot_1,available,None
2,truck_3,rebalancing_truck,depot_1,depot_1,available,None
